# Analysis / Re-plot of FGCM Calibration Information

This notebook is a **standalone analysis notebook** (no Butler access
required) that re-plots the diagnostic figures already shown inline in
`08_AddLSSTFGCM.ipynb`, starting from its saved output table. It is meant to
be re-run quickly to inspect / iterate on the FGCM diagnostics without
re-querying the Butler.

## Figures reproduced

1. **FGCM vs. PhotoCalib zero-point** — scatter comparison, one point per
   (visit, detector), coloured by band.
2. **FGCM zero-point vs. time** (`expMidptMJD`) — one point per
   (visit, detector), coloured by band.
3. **FGCM atmospheric parameters vs. time** — one panel per available
   parameter (PWV, ozone, tau, alpha, ...), coloured by band.
4. **FGCM coverage fraction per band** — bar chart of the fraction of
   light-curve points with atmospheric parameters / a zero-point available.
5. **PhotoCalib vs. FGCM zero-point per band** — 6 stacked subplots
   (`nrows=6, ncols=1`), one per band, each overlaying the PhotoCalib and
   FGCM zero-points vs. MJD on the same axes.

All time-series figures (2, 3, 5) carry a secondary **calendar-date axis**
(`YYYY-MM-DD`) on top of the MJD axis, since MJD and matplotlib date numbers
are both linear day counts differing only by a fixed offset.

## Input
- `data_AddFGCM_08_out/all_stars_lightcurves_fgcm.csv`

## Output
- `figs_AnaFGCM_09/*.pdf` + `.png`

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-07-01
- **Last update:** 2026-07-01


## 1. Imports

In [ ]:
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

from astropy.time import Time

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found -> interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found -> %matplotlib inline")

## 2. Logging setup

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging initialised.")

## 3. Configuration

In [ ]:
# ── Notebook tag ─────────────────────────────────────────────────────────────
NB_TAG = "AnaFGCM_09"

# ── Input: FGCM-enriched light curves (output of notebook 08) ────────────────
DIR_DATA_IN = "./data_AddFGCM_08_out"
GLOBAL_FGCM_FILE = "all_stars_lightcurves_fgcm.csv"

# ── Output ────────────────────────────────────────────────────────────────────
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Directory ready: %s", DIR_FIGS)

# ── Band order (LSST community convention) ──────────────────────────────────
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]

log.info("Configuration done.")

## 4. Load the FGCM-enriched light-curve table

In [ ]:
data_path = os.path.join(DIR_DATA_IN, GLOBAL_FGCM_FILE)
log.info("Loading: %s", data_path)
df_all = pd.read_csv(data_path)
log.info("Shape: %s  |  columns: %s", df_all.shape, df_all.columns.tolist())
df_all.head(3)

In [ ]:
# Verify required columns coming from notebook 08.
REQUIRED_COLS = [
    "band",
    "visit",
    "detector_id",
    "fgcm_zeropoint",
    "fgcm_atm_flag",
    "fgcm_zpt_flag",
    "fgcm_flag",
]
missing = [c for c in REQUIRED_COLS if c not in df_all.columns]
if missing:
    raise ValueError(f"Required columns missing from input FGCM file: {missing}")
log.info("All required columns present.")

# Atmospheric parameter columns: any column starting with 'fgcm_' that is not
# one of the zero-point / flag columns handled separately.
NON_ATM_FGCM_COLS = {"fgcm_zeropoint", "fgcm_zeropoint_err", "fgcm_atm_flag", "fgcm_zpt_flag", "fgcm_flag"}
fgcm_atm_cols = [c for c in df_all.columns if c.startswith("fgcm_") and c not in NON_ATM_FGCM_COLS]
log.info("Atmospheric-parameter columns detected: %s", fgcm_atm_cols)

## 5. Bands present in the data

In [ ]:
bands_available = [b for b in BAND_ORDER if b in df_all["band"].unique()]
extra_bands = sorted(set(df_all["band"].dropna().unique()) - set(BAND_ORDER))
if extra_bands:
    log.warning("Unexpected bands found (not in standard ugrizy list): %s", extra_bands)
    bands_available += extra_bands

n_bands = len(bands_available)
cmap = plt.get_cmap("tab10")

log.info("Bands to plot (%d): %s", n_bands, bands_available)
df_all["band"].value_counts().reindex(bands_available)

## 6. FGCM coverage summary per band

In [ ]:
coverage_summary = df_all.groupby("band").agg(
    n_rows=("fgcm_flag", "size"),
    n_atm_ok=("fgcm_atm_flag", "sum"),
    n_zpt_ok=("fgcm_zpt_flag", "sum"),
    n_any_ok=("fgcm_flag", "sum"),
)
coverage_summary["frac_atm_ok"] = coverage_summary["n_atm_ok"] / coverage_summary["n_rows"]
coverage_summary["frac_zpt_ok"] = coverage_summary["n_zpt_ok"] / coverage_summary["n_rows"]
print("FGCM coverage per band:")
display(coverage_summary)

In [ ]:
visits_all = df_all["visit"].unique()
visits_with_zpt = df_all.loc[df_all["fgcm_zpt_flag"], "visit"].unique()
log.info(
    "Distinct visits: %d total | %d with an FGCM zero-point (%.1f%%) | %d without.",
    len(visits_all),
    len(visits_with_zpt),
    100.0 * len(visits_with_zpt) / len(visits_all) if len(visits_all) else 0,
    len(visits_all) - len(visits_with_zpt),
)

## 7. Plot utilities

In [ ]:
def savefig(fig, name, dpi=150):
    """Save figure as both PDF and PNG."""
    fig.savefig(os.path.join(DIR_FIGS, name + ".pdf"), dpi=dpi, bbox_inches="tight")
    fig.savefig(os.path.join(DIR_FIGS, name + ".png"), dpi=dpi, bbox_inches="tight")
    log.info("Figure saved: %s (.pdf/.png)", os.path.join(DIR_FIGS, name))

In [ ]:
# MJD and matplotlib date numbers are both linear day counts (they only differ
# by a fixed offset), so converting between them is a simple additive shift.
# This lets us add a calendar-date axis on top of an MJD axis "for free",
# without needing a nonlinear/categorical secondary axis.
MJD_TO_MPL_OFFSET = mdates.date2num(Time(0.0, format="mjd").to_datetime())


def _mjd_to_mpl(mjd):
    return np.asarray(mjd, dtype=float) + MJD_TO_MPL_OFFSET


def _mpl_to_mjd(mpl):
    return np.asarray(mpl, dtype=float) - MJD_TO_MPL_OFFSET


def add_date_top_axis(ax):
    """Add a secondary top x-axis showing calendar dates (YYYY-MM-DD),
    derived from the bottom MJD axis of *ax*.
    """
    secax = ax.secondary_xaxis("top", functions=(_mjd_to_mpl, _mpl_to_mjd))
    secax.xaxis.set_major_locator(mdates.AutoDateLocator())
    secax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    plt.setp(secax.get_xticklabels(), rotation=45, ha="left", fontsize=7)
    return secax

## 8. Figure 1 — FGCM vs. PhotoCalib zero-point

In [ ]:
if "zeropoint" in df_all.columns and n_bands > 0 and df_all["fgcm_zpt_flag"].any():
    fig, ax = plt.subplots(figsize=(6, 6))
    for band in bands_available:
        sub = df_all[
            (df_all["band"] == band) & df_all["fgcm_zeropoint"].notna() & df_all["zeropoint"].notna()
        ].drop_duplicates(subset=["visit", "detector_id"])
        if len(sub) == 0:
            continue
        color = cmap(bands_available.index(band))
        ax.scatter(sub["zeropoint"], sub["fgcm_zeropoint"], s=8, alpha=0.5, label=band, color=color)
    lims = df_all[["zeropoint", "fgcm_zeropoint"]].dropna()
    if len(lims) > 0:
        lo, hi = lims.min().min(), lims.max().max()
        ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="1:1")
    ax.set_xlabel("PhotoCalib zero-point  [AB mag]  (notebook 05)")
    ax.set_ylabel("FGCM zero-point  [AB mag]")
    ax.set_title("FGCM vs. PhotoCalib zero-point — one point per (visit, detector)")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    savefig(fig, "fgcm_vs_photocalib_zeropoint")
    plt.show()
else:
    log.info("Skipping FGCM-vs-PhotoCalib zero-point plot (no data).")

## 9. Figure 2 — FGCM zero-point vs. time

In [ ]:
if "expMidptMJD" in df_all.columns and n_bands > 0 and df_all["fgcm_zpt_flag"].any():
    fig, ax = plt.subplots(figsize=(12, 5))
    for band in bands_available:
        sub = df_all[(df_all["band"] == band) & df_all["fgcm_zeropoint"].notna()].drop_duplicates(
            subset=["visit", "detector_id"]
        )
        if len(sub) == 0:
            continue
        color = cmap(bands_available.index(band))
        ax.scatter(sub["expMidptMJD"], sub["fgcm_zeropoint"], s=10, alpha=0.6, label=band, color=color)
    ax.set_xlabel("expMidptMJD")
    ax.set_ylabel("FGCM zero-point  [AB mag]")
    ax.set_title("FGCM zero-point vs. time — one point per (visit, detector)")
    ax.legend(loc="best", fontsize=8)
    add_date_top_axis(ax)
    plt.tight_layout()
    savefig(fig, "fgcm_zeropoint_vs_mjd")
    plt.show()
else:
    log.info("Skipping FGCM zero-point vs. time plot (no data).")

## 10. Figure 3 — FGCM atmospheric parameters vs. time

In [ ]:
if "expMidptMJD" in df_all.columns and fgcm_atm_cols:
    n_params = len(fgcm_atm_cols)
    fig, axes = plt.subplots(n_params, 1, figsize=(12, 2.5 * n_params), sharex=True)
    if n_params == 1:
        axes = [axes]
    for ax, col in zip(axes, fgcm_atm_cols):
        for band in bands_available:
            sub = df_all[(df_all["band"] == band) & df_all[col].notna()].drop_duplicates(subset=["visit"])
            if len(sub) == 0:
                continue
            color = cmap(bands_available.index(band))
            ax.scatter(sub["expMidptMJD"], sub[col], s=15, alpha=0.5, label=band, color=color)
        ax.set_ylabel(col)
        ax.legend(loc="best", fontsize=7, ncol=n_bands)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel("expMidptMJD")
    add_date_top_axis(axes[0])
    fig.suptitle("FGCM atmospheric parameters vs. time — one point per visit", fontsize=12)
    plt.tight_layout()
    savefig(fig, "fgcm_atm_params_vs_mjd")
    plt.show()
else:
    log.info("Skipping atmospheric-parameter plots (no atmospheric columns available).")

## 11. Figure 4 — FGCM coverage fraction per band

In [ ]:
if n_bands > 0:
    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(n_bands)
    width = 0.35
    frac_atm = coverage_summary.loc[bands_available, "frac_atm_ok"].values
    frac_zpt = coverage_summary.loc[bands_available, "frac_zpt_ok"].values
    ax.bar(x - width / 2, 100 * frac_atm, width, label="atmospheric params")
    ax.bar(x + width / 2, 100 * frac_zpt, width, label="zero-point")
    ax.set_xticks(x)
    ax.set_xticklabels(bands_available)
    ax.set_ylabel("FGCM coverage  [%]")
    ax.set_title("Fraction of light-curve points with an FGCM match, per band")
    ax.legend(loc="best", fontsize=8)
    ax.set_ylim(0, 105)
    plt.tight_layout()
    savefig(fig, "fgcm_coverage_fraction_per_band")
    plt.show()

## 12. Figure 5 — PhotoCalib vs. FGCM zero-point per band (6-panel)

One figure, 6 stacked subplots (`nrows=6, ncols=1`), one per band in the
standard `u, g, r, i, z, y` order. Each subplot overlays, vs. MJD:

- the **PhotoCalib zero-point** (`zeropoint`, from notebook 05), and
- the **FGCM zero-point** (`fgcm_zeropoint`, from notebook 08)

on the same axes, so any systematic offset or trend between the two
calibrations is directly visible per band. A calendar-date axis
(`YYYY-MM-DD`) is added on top of the top-most subplot.


In [ ]:
def plot_zeropoint_comparison_per_band(df: pd.DataFrame, band_order: list, dir_figs: str) -> None:
    """6 stacked subplots (nrows=6, ncols=1), one per band: PhotoCalib
    zero-point and FGCM zero-point vs. expMidptMJD on the same axes, with a
    calendar-date axis on top of the top-most subplot.
    """
    has_photocalib = "zeropoint" in df.columns
    has_fgcm = "fgcm_zeropoint" in df.columns
    if not (has_photocalib or has_fgcm) or "expMidptMJD" not in df.columns:
        log.info("Skipping per-band zero-point comparison plot (no data).")
        return

    n_rows = len(band_order)
    fig, axes = plt.subplots(n_rows, 1, figsize=(12, 2 * n_rows), sharex=True)
    if n_rows == 1:
        axes = [axes]

    for ax, band in zip(axes, band_order):
        sub_band = df.loc[df["band"] == band]

        if has_photocalib:
            sub_pc = (
                sub_band.dropna(subset=["zeropoint", "expMidptMJD"])
                .drop_duplicates(subset=["visit", "detector_id"])
                .sort_values("expMidptMJD")
            )
            ax.scatter(
                sub_pc["expMidptMJD"],
                sub_pc["zeropoint"],
                s=15,
                alpha=0.5,
                color="tab:blue",
                label="PhotoCalib zero-point",
                marker="o",
            )

        if has_fgcm:
            sub_fgcm = (
                sub_band.dropna(subset=["fgcm_zeropoint", "expMidptMJD"])
                .drop_duplicates(subset=["visit", "detector_id"])
                .sort_values("expMidptMJD")
            )
            ax.scatter(
                sub_fgcm["expMidptMJD"],
                sub_fgcm["fgcm_zeropoint"],
                s=15,
                alpha=0.5,
                color="tab:orange",
                label="FGCM zero-point",
                marker="s",
            )

        n_pts = len(sub_band)
        ax.set_ylabel(f"zero-point [AB mag]\nband '{band}'", fontsize=9)
        ax.set_title(f"band '{band}'  ({n_pts} pts)", fontsize=9, loc="right")
        ax.legend(loc="best", fontsize=7)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("expMidptMJD")
    add_date_top_axis(axes[0])
    fig.suptitle("PhotoCalib vs. FGCM zero-point vs. time, per band", fontsize=13, y=0.995)
    plt.tight_layout()

    out_name = "zeropoint_photocalib_vs_fgcm_per_band"
    savefig(fig, out_name)
    plt.show()


plot_zeropoint_comparison_per_band(df_all, BAND_ORDER, DIR_FIGS)

## 13. Output directory listing

In [ ]:
print(f"\n=== Contents of {DIR_FIGS} ===")
for entry in sorted(os.listdir(DIR_FIGS)):
    full = os.path.join(DIR_FIGS, entry)
    size_kb = os.path.getsize(full) / 1024
    print(f"  [FILE] {entry}  ({size_kb:.1f} kB)")